# Concept Learning, FIND-S and Candidate Elimination
## Machine Learning — BS Computer Science 3(3-0)



| | |
|---|---|
| **Notebook** | `NB03_Concept_Learning.ipynb` |
| **Lecture** | Lectures 4–5 (Week 2) |
| **CLO** | CLO-1 |
| **Dataset** | EnjoySport / Play Tennis (synthetic, defined below) |

**Objective.** See learning as search through a hypothesis space, and understand why keeping many hypotheses beats keeping one.

**By the end of this notebook you will be able to:**

1. Represent hypotheses in conjunctive form and compute the size of the hypothesis space.
2. Implement FIND-S and trace it on EnjoySport.
3. Implement candidate elimination and reproduce the S and G boundaries.
4. Explain what an empty version space tells you.

---

## 0. Setup and reproducibility

In [1]:
# ---------------------------------------------------------------------
# Reproducibility header — required in EVERY notebook you submit
# ---------------------------------------------------------------------
import sys, numpy as np, pandas as pd, sklearn, matplotlib
import matplotlib.pyplot as plt

print('python      ', sys.version.split()[0])
print('numpy       ', np.__version__)
print('pandas      ', pd.__version__)
print('scikit-learn', sklearn.__version__)
print('matplotlib  ', matplotlib.__version__)

RANDOM_STATE = 1
rng = np.random.default_rng(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

python       3.13.7
numpy        2.3.5
pandas       2.3.3
scikit-learn 1.7.2
matplotlib   3.10.7


## 1. The EnjoySport dataset

Six attributes describe each day; the target concept is whether my friend enjoys water sport. This is the classical worked example (Mitchell Ch. 2).

In [2]:
ATTRS = ['Sky', 'AirTemp', 'Humidity', 'Wind', 'Water', 'Forecast']
VALUES = {'Sky': ['Sunny', 'Rainy', 'Cloudy'],
          'AirTemp': ['Warm', 'Cold'],
          'Humidity': ['Normal', 'High'],
          'Wind': ['Strong', 'Weak'],
          'Water': ['Warm', 'Cool'],
          'Forecast': ['Same', 'Change']}

examples = [
    (('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'), 'Yes'),
    (('Sunny', 'Warm', 'High',   'Strong', 'Warm', 'Same'), 'Yes'),
    (('Rainy', 'Cold', 'High',   'Strong', 'Warm', 'Change'), 'No'),
    (('Sunny', 'Warm', 'High',   'Strong', 'Cool', 'Change'), 'Yes'),
]

display(pd.DataFrame([list(x) + [y] for x, y in examples],
                     columns=ATTRS + ['EnjoySport']))

,Sky,AirTemp,Humidity,Wind,Water,Forecast,EnjoySport
0,Sunny,Warm,Normal,Strong,Warm,Same,Yes
1,Sunny,Warm,High,Strong,Warm,Same,Yes
2,Rainy,Cold,High,Strong,Warm,Change,No
3,Sunny,Warm,High,Strong,Cool,Change,Yes


## 2. How big is the search space?

Compute both numbers by hand first, then check with code. The comparison is the point of the lecture.

In [3]:
n_instances = 1
for a in ATTRS:
    n_instances *= len(VALUES[a])

n_hypotheses = 1
for a in ATTRS:
    n_hypotheses *= (len(VALUES[a]) + 1)      # +1 for '?'
n_hypotheses += 1                              # all the 0-containing ones collapse to one

print('distinct instances                :', n_instances)
print('semantically distinct hypotheses  :', n_hypotheses)
print('boolean functions over instances  : 2^%d = about 10^%d'
      % (n_instances, round(n_instances * np.log10(2))))

distinct instances                : 96
semantically distinct hypotheses  : 973
boolean functions over instances  : 2^96 = about 10^29


> **Inductive bias.** This learner can express 973 of roughly 10^29 possible concepts. It *structurally cannot* represent 'Sunny **or** Cloudy'. That is not a defect: a learner with no bias can fit anything and therefore predicts nothing about unseen data.

## 3. FIND-S — climb from the most specific hypothesis

In [4]:
def find_s(examples, verbose=True):
    """Maximally specific hypothesis consistent with the positive examples."""
    h = None
    for x, label in examples:
        if label != 'Yes':                 # negatives are ignored entirely
            if verbose:
                print('skip (negative):', x)
            continue
        if h is None:
            h = list(x)                    # first positive example
        else:
            for j in range(len(h)):
                if h[j] != x[j]:
                    h[j] = '?'             # minimal generalisation
        if verbose:
            print('after', x, '->', h)
    return h

h = find_s(examples)
print('\nFIND-S result:', h)

after ('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same') -> ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
after ('Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same') -> ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
skip (negative): ('Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change')
after ('Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change') -> ['Sunny', 'Warm', '?', 'Strong', '?', '?']

FIND-S result: ['Sunny', 'Warm', '?', 'Strong', '?', '?']


### Exercise 3.1 — design an example that generalises the result

Add a **fifth positive example** of your own design that makes the final hypothesis strictly more general. State which attribute lost its constraint and why.
The original FIND-S hypothesis is:

⟨Sunny, Warm, ?, Strong, ?, ?⟩

We will add the following fifth positive example:

Sky	AirTemp	Humidity	Wind	Water	Forecast	EnjoySport
Rainy	Warm	Normal	Strong	Warm	Same	Yes
Step 1: Original hypothesis

⟨Sunny, Warm, ?, Strong, ?, ?⟩

Step 2: Compare the new example with the hypothesis

New example:

⟨Rainy, Warm, Normal, Strong, Warm, Same⟩

Original hypothesis:

⟨Sunny, Warm, ?, Strong, ?, ?⟩

The first attribute, Sky, differs:

Original hypothesis: Sunny
New example: Rainy

Therefore, the Sky attribute becomes ?.

Step 3: Final hypothesis

The updated hypothesis is:

⟨?, Warm, ?, Strong, ?, ?⟩

Explanation

The Sky attribute loses its constraint because the new positive example has Rainy instead of Sunny. Since both examples are positive, FIND-S generalises the hypothesis by replacing Sunny with ?.

The final hypothesis is strictly more general because it allows any Sky value while keeping the other existing constraints unchanged.


In [8]:
# TODO: extend `examples` with one new positive example
# TODO: re-run find_s and print the new hypothesis
# TODO: markdown below — which attribute changed, and why?

## 4. Candidate elimination — keep every consistent hypothesis

Positives generalise **S** upward; negatives specialise **G** downward. The version space is everything that survives in between.

In [5]:
def more_general(h1, h2):
    """True if h1 is more general than or equal to h2."""
    for a, b in zip(h1, h2):
        if a == '?':
            continue
        if a != b:
            return False
    return True


def consistent(h, x):
    return all(hi == '?' or hi == xi for hi, xi in zip(h, x))


def candidate_elimination(examples, attrs=ATTRS, values=VALUES, verbose=True):
    S = [tuple('0' for _ in attrs)]                 # most specific
    G = [tuple('?' for _ in attrs)]                 # most general

    for x, label in examples:
        if label == 'Yes':
            G = [g for g in G if consistent(g, x)]
            new_S = []
            for s in S:
                if consistent(s, x):
                    new_S.append(s)
                else:
                    gen = tuple(xi if si == '0' else (si if si == xi else '?')
                                for si, xi in zip(s, x))
                    if any(more_general(g, gen) for g in G):
                        new_S.append(gen)
            S = [s for s in new_S
                 if not any(t != s and more_general(s, t) for t in new_S)]
        else:
            S = [s for s in S if not consistent(s, x)]
            new_G = []
            for g in G:
                if not consistent(g, x):
                    new_G.append(g)
                    continue
                for j, a in enumerate(attrs):        # minimal specialisations
                    if g[j] != '?':
                        continue
                    for v in values[a]:
                        if v == x[j]:
                            continue
                        cand = g[:j] + (v,) + g[j+1:]
                        if any(more_general(cand, s) for s in S):
                            new_G.append(cand)
            G = [g for g in set(new_G)
                 if not any(t != g and more_general(g, t) for t in new_G)]

        if verbose:
            print(f'after {label:>3} {x}')
            print('   S =', S)
            print('   G =', G, '\n')
    return S, G


S, G = candidate_elimination(examples)
print('FINAL  S =', S)
print('FINAL  G =', G)

after Yes ('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same')
   S = [('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same')]
   G = [('?', '?', '?', '?', '?', '?')] 

after Yes ('Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same')
   S = [('Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same')]
   G = [('?', '?', '?', '?', '?', '?')] 

after  No ('Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change')
   S = [('Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same')]
   G = [('Sunny', '?', '?', '?', '?', '?'), ('?', 'Warm', '?', '?', '?', '?'), ('?', '?', '?', '?', '?', 'Same')] 

after Yes ('Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change')
   S = [('Sunny', 'Warm', '?', 'Strong', '?', '?')]
   G = [('Sunny', '?', '?', '?', '?', '?'), ('?', 'Warm', '?', '?', '?', '?')] 

FINAL  S = [('Sunny', 'Warm', '?', 'Strong', '?', '?')]
FINAL  G = [('Sunny', '?', '?', '?', '?', '?'), ('?', 'Warm', '?', '?', '?', '?')]


**Check your trace.** After example 3 (the negative one) `G` should contain exactly three hypotheses. Work out on paper why `⟨?,?,Normal,?,?,?⟩` and `⟨?,?,?,Weak,?,?⟩` are *not* among them.

## 5. What an empty version space means

Flip the label of example 3 from `No` to `Yes` and re-run. Observe what happens to S and G, and explain it.

In [6]:
corrupted = [(x, 'Yes' if i == 2 else lab)
             for i, (x, lab) in enumerate(examples)]

S2, G2 = candidate_elimination(corrupted, verbose=False)
print('S =', S2)
print('G =', G2)

S = [('?', '?', '?', 'Strong', '?', '?')]
G = [('?', '?', '?', '?', '?', '?')]


**Interpretation (two sentences).** An empty version space has exactly two possible causes. Name both, and say which one applies here.

> _Your answer here._

## 6. Play Tennis — where the hypothesis space fails

Run FIND-S on the data below. You should end at `⟨?, ?, ?, ?⟩`, which reads *'always play tennis'* — and that is wrong. Explain why the algorithm is not at fault.

In [7]:
tennis = [
    (('Sunny',    'Hot',  'High',   'Weak'),   'No'),
    (('Overcast', 'Hot',  'High',   'Weak'),   'Yes'),
    (('Rain',     'Mild', 'High',   'Weak'),   'Yes'),
    (('Rain',     'Cool', 'Normal', 'Strong'), 'No'),
    (('Overcast', 'Cool', 'Normal', 'Strong'), 'Yes'),
]

print('FIND-S on Play Tennis:', find_s(tennis, verbose=True))

skip (negative): ('Sunny', 'Hot', 'High', 'Weak')
after ('Overcast', 'Hot', 'High', 'Weak') -> ['Overcast', 'Hot', 'High', 'Weak']
after ('Rain', 'Mild', 'High', 'Weak') -> ['?', '?', 'High', 'Weak']
skip (negative): ('Rain', 'Cool', 'Normal', 'Strong')
after ('Overcast', 'Cool', 'Normal', 'Strong') -> ['?', '?', '?', '?']
FIND-S on Play Tennis: ['?', '?', '?', '?']


---
## Deliverable

Working FIND-S and candidate elimination, the EnjoySport trace reproduced exactly, exercise 3.1 with its written explanation, and the two-sentence interpretation of the empty version space.

---
## Submission checklist

Tick every box before you submit. Items 1–6 are the course reproducibility
rules from Lecture 3 and are marked in the rubric.

- [ ] The random seed is fixed **and printed**
- [ ] Library versions are printed in the first cell
- [ ] The raw data file was **not** modified
- [ ] Preprocessing happens **after** the split, inside a `Pipeline`
- [ ] The dataset source and licence are stated in a markdown cell
- [ ] **Kernel → Restart & Run All** completes without error
- [ ] Every experiment is followed by a written interpretation
- [ ] At least one wrong prediction or failure case is explained
- [ ] Results are reported as **mean ± standard deviation** where cross-validated
- [ ] Every plot has axis labels with units and a legend
